In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None) # for viewing max colmuns
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_parquet('../data/customer churn/merged_data.parquet')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 51 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   customer_id                        7043 non-null   object 
 1   gender                             7043 non-null   object 
 2   age                                7043 non-null   int64  
 3   under_30                           7043 non-null   object 
 4   senior_citizen                     7043 non-null   object 
 5   partner                            7043 non-null   object 
 6   dependents                         7043 non-null   object 
 7   number_of_dependents               7043 non-null   int64  
 8   married                            7043 non-null   object 
 9   country                            7043 non-null   object 
 10  state                              7043 non-null   object 
 11  city                               7043 non-null   objec

In [4]:
info = {
    'shape':df.shape
}
info

{'shape': (7043, 51)}

In [5]:
df.describe()

,age,number_of_dependents,zip_code,total_population,latitude,longitude,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,tenure,avg_monthly_gb_download,number_of_referrals,satisfaction_score,cltv,churn_score,churn_value
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,46.509726,0.468692,93486.071134,22139.814568,36.197455,-119.756684,64.761692,22.958954,2280.381264,1.962182,6.860713,749.099262,3034.379056,32.371149,20.515405,1.951867,3.244924,4400.295755,58.505040,0.265370
std,16.750352,0.962802,1856.768045,21152.174407,2.468929,2.154425,30.090047,15.448113,2266.220462,7.902614,25.104978,846.660055,2865.204542,24.559481,20.418940,3.001199,1.201657,1183.057152,21.170031,0.441561
min,19.000000,0.000000,90001.000000,11.000000,32.555828,-124.301372,18.250000,0.000000,18.800000,0.000000,0.000000,0.000000,21.360000,0.000000,0.000000,0.000000,1.000000,2003.000000,5.000000,0.000000
25%,32.000000,0.000000,92101.000000,2344.000000,33.990646,-121.788090,35.500000,9.210000,400.150000,0.000000,0.000000,70.545000,605.610000,9.000000,3.000000,0.000000,3.000000,3469.000000,40.000000,0.000000
50%,46.000000,0.000000,93518.000000,17554.000000,36.205465,-119.595293,70.350000,22.890000,1394.550000,0.000000,0.000000,401.440000,2108.640000,29.000000,17.000000,0.000000,3.000000,4527.000000,61.000000,0.000000
75%,60.000000,0.000000,95329.000000,36125.000000,38.161321,-117.969795,89.850000,36.395000,3786.600000,0.000000,0.000000,1191.100000,4801.145000,55.000000,27.000000,3.000000,4.000000,5380.500000,75.500000,1.000000
max,80.000000,9.000000,96150.000000,105285.000000,41.962127,-114.192901,118.750000,49.990000,8684.800000,49.790000,150.000000,3564.720000,11979.340000,72.000000,85.000000,11.000000,5.000000,6500.000000,96.000000,1.000000


In [6]:
df.columns[df.isnull().sum()>0].tolist()

['internet_type', 'offer', 'churn_reason']

In [7]:
df['internet_type'].fillna(df.internet_type.mode()[0],axis=0,inplace=True)

In [8]:
df['offer'].fillna(method='ffill',axis=0,inplace=True)
df['offer'].fillna(method='bfill',axis=0,inplace=True)

In [9]:
sus = ['number_of_dependents', 'total_refunds', 'total_extra_data_charges', 'number_of_referrals']

In [10]:
cat = df.select_dtypes(exclude=['int','float']).columns.tolist()

In [11]:
num = df.select_dtypes(include=['int','float']).columns.tolist()

# <span style="color:red"> <b>categorical tunning

In [12]:
df[cat].head()

,customer_id,gender,under_30,senior_citizen,partner,dependents,married,country,state,city,phone_service,internet_service,online_security,online_backup,device_protection,premium_tech_support,streaming_tv,streaming_movies,streaming_music,internet_type,contract,paperless_billing,payment_method,multiple_lines,unlimited_data,offer,referred_a_friend,customer_status,churn_label,churn_category,churn_reason
0,0002-ORFBO,Female,No,No,Yes,No,Yes,United States,California,Frazier Park,Yes,Yes,No,Yes,No,Yes,Yes,No,No,Cable,One Year,Yes,Mailed check,No,Yes,Offer E,Yes,Stayed,No,Not Applicable,None
1,0003-MKNFE,Male,No,No,No,No,No,United States,California,Glendale,Yes,Yes,No,No,No,No,No,Yes,Yes,Cable,Month-to-Month,No,Mailed check,Yes,No,Offer E,No,Stayed,No,Not Applicable,None
2,0004-TLHLJ,Male,No,No,No,No,No,United States,California,Costa Mesa,Yes,Yes,No,No,Yes,No,No,No,No,Fiber Optic,Month-to-Month,Yes,Electronic check,No,Yes,Offer E,No,Churned,Yes,Competitor,Competitor had better devices
3,0011-IGKFF,Male,No,Yes,Yes,No,Yes,United States,California,Martinez,Yes,Yes,No,Yes,Yes,No,Yes,Yes,No,Fiber Optic,Month-to-Month,Yes,Electronic check,No,Yes,Offer D,Yes,Churned,Yes,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,No,Yes,Yes,No,Yes,United States,California,Camarillo,Yes,Yes,No,No,No,Yes,Yes,No,No,Fiber Optic,Month-to-Month,Yes,Mailed check,No,Yes,Offer D,Yes,Churned,Yes,Dissatisfaction,Network reliability


In [13]:
drop =[]
for col in df[cat].columns:
    values = df[col].value_counts()
    print(values)
    print('$%#@'*40,'\n')
    if len(values)==info['shape'][0]: # categorical and all unique values are no use for data modeling, therfore getting names of col with all unique values
        drop.append(col)

customer_id
9995-HOTOH    1
0002-ORFBO    1
0003-MKNFE    1
9938-PRCVK    1
9938-TKDGL    1
             ..
0013-SMEOE    1
0014-BMAQU    1
0015-UOCOJ    1
0016-QLJIS    1
0017-DINOC    1
Name: count, Length: 7043, dtype: int64
$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@ 

gender
Male      3555
Female    3488
Name: count, dtype: int64
$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@ 

under_30
No     5642
Yes    1401
Name: count, dtype: int64
$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@ 

senior_citizen
No     5901
Yes    1142
Name: count, dtype: int64
$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@$%#@

In [14]:
df.drop(drop,axis=1,inplace=True)
for r in drop:
    cat.remove(r)

In [15]:
df.replace(['No','Yes'],[0,1],inplace=True) # this will slove all [No, Yes] - [0,1]

In [16]:
# let's change our cat list
cat = df.select_dtypes(exclude=['int','float']).columns.tolist()
cat

['gender',
 'country',
 'state',
 'city',
 'internet_type',
 'contract',
 'payment_method',
 'offer',
 'customer_status',
 'churn_category',
 'churn_reason']

In [17]:
df[cat]

,gender,country,state,city,internet_type,contract,payment_method,offer,customer_status,churn_category,churn_reason
0,Female,United States,California,Frazier Park,Cable,One Year,Mailed check,Offer E,Stayed,Not Applicable,None
1,Male,United States,California,Glendale,Cable,Month-to-Month,Mailed check,Offer E,Stayed,Not Applicable,None
2,Male,United States,California,Costa Mesa,Fiber Optic,Month-to-Month,Electronic check,Offer E,Churned,Competitor,Competitor had better devices
3,Male,United States,California,Martinez,Fiber Optic,Month-to-Month,Electronic check,Offer D,Churned,Dissatisfaction,Product dissatisfaction
4,Female,United States,California,Camarillo,Fiber Optic,Month-to-Month,Mailed check,Offer D,Churned,Dissatisfaction,Network reliability
...,...,...,...,...,...,...,...,...,...,...,...
7038,Female,United States,California,La Mesa,DSL,One Year,Mailed check,Offer D,Stayed,Not Applicable,None
7039,Male,United States,California,Riverbank,Fiber Optic,Month-to-Month,Electronic check,Offer D,Churned,Dissatisfaction,Product dissatisfaction
7040,Male,United States,California,Elk,DSL,Month-to-Month,Mailed check,Offer E,Joined,Not Applicable,None
7041,Male,United States,California,Solana Beach,Cable,Two Year,Mailed check,Offer A,Stayed,Not Applicable,None


In [18]:
for col in cat:
    print(col,'--->',df[col].nunique(),'\n')

gender ---> 2 

country ---> 1 

state ---> 1 

city ---> 1106 

internet_type ---> 3 

contract ---> 3 

payment_method ---> 4 

offer ---> 5 

customer_status ---> 3 

churn_category ---> 6 

churn_reason ---> 20 



In [19]:
for col in cat:
    print(df[col].value_counts(), '\n')

gender
Male      3555
Female    3488
Name: count, dtype: int64 

country
United States    7043
Name: count, dtype: int64 

state
California    7043
Name: count, dtype: int64 

city
Los Angeles         293
San Diego           285
San Jose            112
Sacramento          108
San Francisco       104
                   ... 
Eldridge              2
Jacumba               2
South Lake Tahoe      2
Johannesburg          2
Holtville             2
Name: count, Length: 1106, dtype: int64 

internet_type
Fiber Optic    4561
DSL            1652
Cable           830
Name: count, dtype: int64 

contract
Month-to-Month    3610
Two Year          1883
One Year          1550
Name: count, dtype: int64 

payment_method
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64 

offer
Offer B    1812
Offer E    1758
Offer D    1330
Offer A    1226
Offer C     917
Name: count, dtype: int64 

customer_sta

In [20]:
drop_x = ['churn_reason', 'churn_category', 'customer_status', 'city'] #first 3 are dublicate cols or high unique values can't convert them to numeric
df_x=df[drop_x] # storing them if needs in future
df.drop(drop_x, axis=1, inplace=True) #drop
for r in drop_x: #removing from cat list
    cat.remove(r)

In [21]:
cat

['gender',
 'country',
 'state',
 'internet_type',
 'contract',
 'payment_method',
 'offer']

In [22]:
col = 'gender'
uni_index = df.gender.value_counts().index.tolist()
map_values = [i+1 for i in range(len(uni_index))]


for col in cat:
    uni_index = df[col].value_counts().index.tolist()
    map_values = [i+1 for i in range(len(uni_index))]
    df.replace(uni_index,map_values,inplace=True)
    info.update({col:{k:v for k,v in zip(map_values,uni_index)}})

In [23]:
info

{'shape': (7043, 51),
 'gender': {1: 'Male', 2: 'Female'},
 'country': {1: 'United States'},
 'state': {1: 'California'},
 'internet_type': {1: 'Fiber Optic', 2: 'DSL', 3: 'Cable'},
 'contract': {1: 'Month-to-Month', 2: 'Two Year', 3: 'One Year'},
 'payment_method': {1: 'Electronic check',
  2: 'Mailed check',
  3: 'Bank transfer (automatic)',
  4: 'Credit card (automatic)'},
 'offer': {1: 'Offer B',
  2: 'Offer E',
  3: 'Offer D',
  4: 'Offer A',
  5: 'Offer C'}}

In [24]:
# let see if we still have cat ?
cat = df.select_dtypes(exclude=['int','float']).columns.tolist()
cat
# cat is None

[]

# <span style = 'color:red'> <b>Numeric tunning

In [25]:
num

['age',
 'number_of_dependents',
 'zip_code',
 'total_population',
 'latitude',
 'longitude',
 'monthly_ charges',
 'avg_monthly_long_distance_charges',
 'total_charges',
 'total_refunds',
 'total_extra_data_charges',
 'total_long_distance_charges',
 'total_revenue',
 'tenure',
 'avg_monthly_gb_download',
 'number_of_referrals',
 'satisfaction_score',
 'cltv',
 'churn_score',
 'churn_value']

In [26]:
df[sus].describe() # all sus columns will always be numeric

,number_of_dependents,total_refunds,total_extra_data_charges,number_of_referrals
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.468692,1.962182,6.860713,1.951867
std,0.962802,7.902614,25.104978,3.001199
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,3.000000
max,9.000000,49.790000,150.000000,11.000000


In [27]:
df.number_of_dependents.value_counts()

number_of_dependents
0    5416
1     553
2     531
3     517
5      10
4       9
6       3
7       2
9       1
8       1
Name: count, dtype: int64

### <span style = 'color:lime'>pretty much skewed 

In [28]:
df[df['number_of_dependents']>3].shape

(26, 46)

#### <span style='color:orange'> we will only lose 26 row so let's delete all them

In [29]:
df.drop(df[df['number_of_dependents']>3].index,axis=0, inplace=True)

In [30]:
df.total_refunds.value_counts()

total_refunds
0.00     6494
12.48       2
5.73        2
8.74        2
16.56       2
         ... 
16.91       1
23.16       1
17.76       1
24.98       1
13.37       1
Name: count, Length: 498, dtype: int64

### <span style = 'color:lime'> this also skewed almost all have 0.0 refund so let's drop this col

In [31]:
df.drop('total_refunds',axis=1, inplace=True)
num.remove('total_refunds')

In [32]:
df.total_extra_data_charges.value_counts()

total_extra_data_charges
0      6292
10      137
40       62
30       57
20       51
80       47
100      44
50       43
150      42
130      40
140      38
60       36
90       35
70       34
110      31
120      28
Name: count, dtype: int64

In [33]:
df[df.total_extra_data_charges>0]

,gender,age,under_30,senior_citizen,partner,dependents,number_of_dependents,married,country,state,zip_code,total_population,latitude,longitude,phone_service,internet_service,online_security,online_backup,device_protection,premium_tech_support,streaming_tv,streaming_movies,streaming_music,internet_type,contract,paperless_billing,payment_method,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_extra_data_charges,total_long_distance_charges,total_revenue,tenure,multiple_lines,avg_monthly_gb_download,unlimited_data,offer,referred_a_friend,number_of_referrals,satisfaction_score,cltv,churn_score,churn_label,churn_value
1,1,46,0,0,0,0,0,0,1,1,91206,31297,34.162515,-118.203869,1,1,0,0,0,0,0,1,1,3,1,0,2,59.90,10.69,542.40,10,96.21,610.28,9,1,10,0,2,0,0,5,5414,66,0,0
7,1,52,0,0,1,0,0,1,1,1,94558,63947,38.489789,-122.270110,1,1,1,0,0,1,0,0,0,1,2,1,4,84.65,12.96,5377.80,20,816.48,6214.28,63,1,7,0,1,1,8,4,4604,49,0,0
26,2,37,0,0,1,1,3,1,1,1,93648,12587,36.622237,-119.521126,1,1,0,1,1,1,1,1,1,1,3,1,3,103.70,35.04,5656.75,20,1927.20,7603.95,55,0,57,0,2,1,10,3,5892,65,0,0
49,1,50,0,0,0,0,0,0,1,1,95230,596,37.956963,-120.863055,1,1,0,0,0,1,1,1,1,1,1,1,1,94.20,2.05,2607.60,20,59.45,2687.05,29,0,17,0,2,0,0,5,4561,60,0,0
50,1,25,1,0,0,0,0,0,1,1,96134,2595,41.813521,-121.492666,1,1,1,1,1,1,1,0,0,2,3,0,1,81.25,5.49,5567.55,40,351.36,5958.91,64,1,69,0,2,0,0,3,4761,67,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6994,2,45,0,0,1,0,0,1,1,1,93631,14088,36.478239,-119.521370,1,1,0,0,1,0,1,1,1,3,2,1,3,76.80,43.47,5468.45,90,3129.84,8688.29,72,1,9,0,4,1,5,3,5120,34,0,0
7013,1,44,0,0,1,0,0,1,1,1,94903,28403,38.018065,-122.546024,1,1,1,1,0,0,0,0,0,1,1,1,1,79.45,35.61,3013.05,40,1353.18,4396.87,38,0,13,0,3,1,1,1,2405,85,1,1
7017,1,75,0,1,1,0,0,1,1,1,96015,417,41.486953,-120.913975,1,1,0,0,1,0,0,0,0,1,2,1,1,80.80,16.87,457.10,30,101.22,588.32,6,1,17,0,1,1,4,5,5611,68,0,0
7027,1,70,0,1,1,1,1,1,1,1,94563,17964,37.873916,-122.205220,1,1,0,1,1,1,1,1,1,1,1,1,1,108.90,11.74,3625.20,30,399.16,4054.36,34,1,12,0,2,1,7,3,5151,24,0,0


#### <span style = 'color:orange'> this is 725 almost a 10% so let's divide this col into [0,1] 0 who paid nothing, 1 for who paid more

In [34]:
df['extra_data_charges'] = df['total_extra_data_charges'].apply(lambda x:0 if x==0 else 1)
df.drop('total_extra_data_charges',axis=1,inplace=True)
num.remove('total_extra_data_charges')

In [35]:
df.extra_data_charges.value_counts()

extra_data_charges
0    6292
1     725
Name: count, dtype: int64

In [36]:
info.update({'extra_data_charges':{0:'not_paid',1:'paid'}})

In [37]:
df.number_of_referrals.value_counts()

number_of_referrals
0     3816
1     1082
5      261
3      254
7      247
4      236
9      235
2      233
10     221
6      219
8      211
11       2
Name: count, dtype: int64

### <span style='color:lime'> let's do same to this divide [0,1,2] o for not referred, 1 for referred and 2 for referred more than 5 times

In [38]:
df['referrals'] = df['number_of_referrals'].apply(lambda x:0 if x==0 else (1 if 0<x<5 else 2))
df.drop('number_of_referrals',axis=1,inplace=True)
num.remove('number_of_referrals')

In [39]:
df.referrals.value_counts()

referrals
0    3816
1    1805
2    1396
Name: count, dtype: int64

In [40]:
info.update({'referrals':{0:'not_referred',1:'referred_less_than_5',2:'referred_more_than_5'}})

In [41]:
sus = None
#sus is None

In [42]:
#now let's go to num
df[num]

,age,number_of_dependents,zip_code,total_population,latitude,longitude,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_long_distance_charges,total_revenue,tenure,avg_monthly_gb_download,satisfaction_score,cltv,churn_score,churn_value
0,37,0,93225,4498,34.827662,-118.999073,65.60,42.39,593.30,381.51,974.81,9,16,3,2205,65,0
1,46,0,91206,31297,34.162515,-118.203869,59.90,10.69,542.40,96.21,610.28,9,10,5,5414,66,0
2,50,0,92627,62069,33.645672,-117.922613,73.90,33.65,280.85,134.60,415.45,4,30,1,4479,71,1
3,78,0,94553,46677,38.014457,-122.115432,98.00,27.82,1237.85,361.66,1599.51,13,4,1,3714,91,1
4,75,0,93010,42853,34.227846,-119.079903,83.90,7.38,267.40,22.14,289.54,3,11,1,3464,68,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,20,0,91941,44652,32.759327,-116.997260,55.15,46.68,742.90,606.84,1349.74,13,59,4,3161,59,0
7039,40,0,95367,16525,37.734971,-120.954271,85.10,16.20,1873.70,356.40,2230.10,22,17,1,5248,68,1
7040,22,0,95432,383,39.108252,-123.645121,50.30,18.62,92.75,37.24,129.99,2,51,5,5870,33,0
7041,21,0,92075,12173,33.001813,-117.263628,67.85,2.12,4627.65,142.04,4769.69,67,58,3,4792,59,0


##### <span style='color:orange'> what's the need of latitude, longitude, zip_code

In [43]:
df.drop(['latitude', 'longitude','zip_code'], axis=1, inplace=True)

In [44]:
df

,gender,age,under_30,senior_citizen,partner,dependents,number_of_dependents,married,country,state,total_population,phone_service,internet_service,online_security,online_backup,device_protection,premium_tech_support,streaming_tv,streaming_movies,streaming_music,internet_type,contract,paperless_billing,payment_method,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_long_distance_charges,total_revenue,tenure,multiple_lines,avg_monthly_gb_download,unlimited_data,offer,referred_a_friend,satisfaction_score,cltv,churn_score,churn_label,churn_value,extra_data_charges,referrals
0,2,37,0,0,1,0,0,1,1,1,4498,1,1,0,1,0,1,1,0,0,3,3,1,2,65.60,42.39,593.30,381.51,974.81,9,0,16,1,2,1,3,2205,65,0,0,0,1
1,1,46,0,0,0,0,0,0,1,1,31297,1,1,0,0,0,0,0,1,1,3,1,0,2,59.90,10.69,542.40,96.21,610.28,9,1,10,0,2,0,5,5414,66,0,0,1,0
2,1,50,0,0,0,0,0,0,1,1,62069,1,1,0,0,1,0,0,0,0,1,1,1,1,73.90,33.65,280.85,134.60,415.45,4,0,30,1,2,0,1,4479,71,1,1,0,0
3,1,78,0,1,1,0,0,1,1,1,46677,1,1,0,1,1,0,1,1,0,1,1,1,1,98.00,27.82,1237.85,361.66,1599.51,13,0,4,1,3,1,1,3714,91,1,1,0,1
4,2,75,0,1,1,0,0,1,1,1,42853,1,1,0,0,0,1,1,0,0,1,1,1,2,83.90,7.38,267.40,22.14,289.54,3,0,11,1,3,1,1,3464,68,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,2,20,1,0,0,0,0,0,1,1,44652,1,1,1,0,0,1,0,0,1,2,3,0,2,55.15,46.68,742.90,606.84,1349.74,13,0,59,1,3,0,4,3161,59,0,0,0,0
7039,1,40,0,0,1,0,0,1,1,1,16525,1,1,0,0,0,0,0,1,1,1,1,1,1,85.10,16.20,1873.70,356.40,2230.10,22,1,17,1,3,1,1,5248,68,1,1,0,1
7040,1,22,1,0,0,0,0,0,1,1,383,1,1,0,1,0,0,0,0,0,2,1,1,2,50.30,18.62,92.75,37.24,129.99,2,0,51,1,2,0,5,5870,33,0,0,0,0
7041,1,21,1,0,1,0,0,1,1,1,12173,1,1,1,0,1,1,0,1,1,3,2,0,2,67.85,2.12,4627.65,142.04,4769.69,67,0,58,1,4,1,3,4792,59,0,0,0,2


In [45]:
info.update({'clean_data':df.shape})

In [46]:
# import json
# with open('note.txt', 'w') as file:
#     file.write(json.dumps(info,indent=4))

print('done')

In [49]:

# import logging

# # This cell
# logger = logging.getLogger('my_notebook_logger')
# logger.setLevel(logging.INFO)

# if not logger.handlers:  # The crucial check
#     fh = logging.FileHandler('notebook.log')
#     formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
#     fh.setFormatter(formatter)
#     logger.addHandler(fh)

# logger.info(json.dumps(info,indent=4))

print('done')

In [50]:
df

,gender,age,under_30,senior_citizen,partner,dependents,number_of_dependents,married,country,state,total_population,phone_service,internet_service,online_security,online_backup,device_protection,premium_tech_support,streaming_tv,streaming_movies,streaming_music,internet_type,contract,paperless_billing,payment_method,monthly_ charges,avg_monthly_long_distance_charges,total_charges,total_long_distance_charges,total_revenue,tenure,multiple_lines,avg_monthly_gb_download,unlimited_data,offer,referred_a_friend,satisfaction_score,cltv,churn_score,churn_label,churn_value,extra_data_charges,referrals
0,2,37,0,0,1,0,0,1,1,1,4498,1,1,0,1,0,1,1,0,0,3,3,1,2,65.60,42.39,593.30,381.51,974.81,9,0,16,1,2,1,3,2205,65,0,0,0,1
1,1,46,0,0,0,0,0,0,1,1,31297,1,1,0,0,0,0,0,1,1,3,1,0,2,59.90,10.69,542.40,96.21,610.28,9,1,10,0,2,0,5,5414,66,0,0,1,0
2,1,50,0,0,0,0,0,0,1,1,62069,1,1,0,0,1,0,0,0,0,1,1,1,1,73.90,33.65,280.85,134.60,415.45,4,0,30,1,2,0,1,4479,71,1,1,0,0
3,1,78,0,1,1,0,0,1,1,1,46677,1,1,0,1,1,0,1,1,0,1,1,1,1,98.00,27.82,1237.85,361.66,1599.51,13,0,4,1,3,1,1,3714,91,1,1,0,1
4,2,75,0,1,1,0,0,1,1,1,42853,1,1,0,0,0,1,1,0,0,1,1,1,2,83.90,7.38,267.40,22.14,289.54,3,0,11,1,3,1,1,3464,68,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,2,20,1,0,0,0,0,0,1,1,44652,1,1,1,0,0,1,0,0,1,2,3,0,2,55.15,46.68,742.90,606.84,1349.74,13,0,59,1,3,0,4,3161,59,0,0,0,0
7039,1,40,0,0,1,0,0,1,1,1,16525,1,1,0,0,0,0,0,1,1,1,1,1,1,85.10,16.20,1873.70,356.40,2230.10,22,1,17,1,3,1,1,5248,68,1,1,0,1
7040,1,22,1,0,0,0,0,0,1,1,383,1,1,0,1,0,0,0,0,0,2,1,1,2,50.30,18.62,92.75,37.24,129.99,2,0,51,1,2,0,5,5870,33,0,0,0,0
7041,1,21,1,0,1,0,0,1,1,1,12173,1,1,1,0,1,1,0,1,1,3,2,0,2,67.85,2.12,4627.65,142.04,4769.69,67,0,58,1,4,1,3,4792,59,0,0,0,2


In [51]:
scale = ['cltv', 'avg_monthly_gb_download', 'total_revenue', 'tenure', 'total_long_distance_charges', 'total_charges', 'avg_monthly_long_distance_charges', 'monthly_ charges', 'total_population']
df[scale].describe()

,cltv,avg_monthly_gb_download,total_revenue,tenure,total_long_distance_charges,total_charges,avg_monthly_long_distance_charges,monthly_ charges,total_population
count,7017.000000,7017.000000,7017.000000,7017.000000,7017.000000,7017.000000,7017.000000,7017.000000,7017.000000
mean,4400.972638,20.464158,3034.365250,32.376087,748.473794,2280.988377,22.947818,64.798012,22119.877441
std,1182.359918,20.377681,2864.169304,24.562426,846.138263,2265.754011,15.445398,30.083276,21153.203387
min,2003.000000,0.000000,21.360000,0.000000,0.000000,18.800000,0.000000,18.250000,11.000000
25%,3470.000000,3.000000,605.800000,9.000000,70.650000,400.000000,9.210000,35.600000,2318.000000
50%,4527.000000,17.000000,2108.650000,29.000000,401.100000,1396.250000,22.880000,70.350000,17494.000000
75%,5380.000000,27.000000,4800.360000,55.000000,1189.440000,3789.200000,36.390000,89.850000,36123.000000
max,6500.000000,85.000000,11979.340000,72.000000,3564.720000,8684.800000,49.990000,118.750000,105285.000000


In [53]:
# df.to_parquet('../data/customer churn/clean_data.parquet')
print("data saved!")

data saved!
